In [ ]:
# Reinforcement Learning Basics: Q-Learning on CartPole Environment

# Import necessary libraries
import numpy as np
import gym
import random
import matplotlib.pyplot as plt

# Step 1: Create the environment
env = gym.make('CartPole-v1')

# Step 2: Initialize Q-table
n_actions = env.action_space.n
n_states = np.prod(env.observation_space.shape)  # Continuous state space
q_table = np.zeros((n_states, n_actions))  # Initialize Q-table with zeros

# Step 3: Define hyperparameters
alpha = 0.1  # Learning rate
gamma = 0.99  # Discount factor
epsilon = 0.1  # Exploration rate
episodes = 1000  # Number of episodes

# Step 4: Discretize the state space
def discretize_state(state):
    """
    Discretizes the continuous state space into bins for Q-learning.
    """
    return tuple(np.digitize(state, bins=np.linspace(-1.2, 1.2, num=10)))

# Step 5: Training loop
for episode in range(episodes):
    state = env.reset()
    state = discretize_state(state)
    
    done = False
    total_reward = 0
    
    while not done:
        # Step 5.1: Choose an action (Explore or Exploit)
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample()  # Explore
        else:
            action = np.argmax(q_table[state])  # Exploit
            
        # Step 5.2: Take the action and observe the next state and reward
        next_state, reward, done, _ = env.step(action)
        next_state = discretize_state(next_state)
        
        # Step 5.3: Update Q-table using the Q-learning formula
        q_table[state][action] = q_table[state][action] + alpha * (
            reward + gamma * np.max(q_table[next_state]) - q_table[state][action]
        )
        
        # Update the current state and accumulate the reward
        state = next_state
        total_reward += reward
    
    # Print progress every 100 episodes
    if episode % 100 == 0:
        print(f"Episode {episode}, Total Reward: {total_reward}")

# Step 6: Save the Q-table
np.save('q_table.npy', q_table)
print("Q-table saved as 'q_table.npy'.")

# Step 7: Test the trained agent
state = env.reset()
state = discretize_state(state)
done = False
print("Testing the trained agent...")
while not done:
    action = np.argmax(q_table[state])  # Choose action based on learned policy
    state, reward, done, _ = env.step(action)
    env.render()  # Render the environment
env.close()